# Lesson 03 - エージェントデザインパターン

このレッスンでは、効果的なAIエージェントを構築するための3つの基本的なデザインパターンを探ります：

1. **明確なエージェント指示** — エージェントの行動を導くための正確で役割を定義するプロンプトの作成
2. **Pydanticモデルによる構造化された出力** — エージェントが予測可能で検証済みのデータを返すことを保証
3. **単一責任のエージェント** — 各々が一つのことをうまく行うフォーカスされたエージェントの設計

これらのパターンを、**旅行先推薦システム**のシナリオに適用し、目的地の提案、空き状況の確認、ロジスティクスの処理を段階的に構築していきます。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## パターン1: 明確なエージェント指示

最も効果的なパターンは、最もシンプルなものでもあります。それは、エージェントへの明確で詳細な指示を書くことです。

良い指示は以下を定義します：
- **誰**がエージェントなのか（ペルソナとトーン）
- **何**をすべきか（ステップバイステップの役割）
- **どのように**振る舞うべきか（制約とスタイル）

以下に、すべての応答を形作る明確な指示を持つ旅行コンシェルジュエージェントを作成します。


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## パターン 2: Pydantic モデルによる構造化出力

自由形式のテキストは会話には便利ですが、下流のシステムでは構造化データが必要です。  
**Pydantic モデル**と**ツール関数**を組み合わせることで、以下が可能になります:

- エージェントの出力に対して正確なスキーマを定義する
- 応答の自動検証を行う
- エージェントの結果をアプリケーションロジックに確実に統合する

また、エージェントが推奨を実際のデータに基づいて行えるように、目的地の詳細を返すツールも紹介します。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## パターン3: 単一責任エージェント

複雑なタスクは、単一の責任を持つ複数の専門化されたエージェントに作業を分割することで効果が上がります:

- 場所や空き状況に詳しい**目的地エキスパート**
- フライト、ホテル、旅程を担当する**物流プランナー**

これはソフトウェア工学の原則である*関心の分離*と対応しており、各エージェントは独立してテスト、メンテナンス、および改良が容易になります。


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

このレッスンでは、旅行レコメンダーのシナリオに対して3つのエージェント設計パターンを適用しました：

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | ペルソナ、責任、制約を事前に定義する | 一貫性のあるブランドに沿ったエージェントの動作 |
| **Structured Output** | Pydanticモデルを応答フォーマットとして使用する | 検証済みでマシンリーダブルな結果 |
| **Single Responsibility** | 各エージェントに一つの集中した役割を与える | テスト、保守、組み合わせが容易になる |

これらのパターンは自然に組み合わせられます — 明確な指示と構造化された出力を単一責任のエージェント内で組み合わせて、堅牢で本番対応のシステムを構築できます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類は、AI翻訳サービス[Co-op Translator](https://github.com/Azure/co-op-translator)を用いて翻訳されました。正確性には努めておりますが、自動翻訳には誤りや不正確な箇所が含まれる場合があります。原文が正式かつ権威ある情報源とみなされます。重要な情報については、専門の人間による翻訳を推奨いたします。本翻訳の利用により生じたいかなる誤解や誤訳についても、当方は一切責任を負いかねますのでご了承ください。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
